# 第4章 智能系统深度学习开发

> **课程章节**：第4章 智能系统深度学习开发
> **运行环境**：cann_9.0.0-py3.11-A2-arm-20260715 | ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB
> **建议学时**：4 学时

本 Notebook 是第4章的配套实践教程，将带你从理论到代码，全面理解深度学习开发的核心概念，并在昇腾 NPU 上亲手运行代码。

## 学习目标

1. 理解主流深度学习网络架构（CNN、RNN、Transformer）的设计原理与适用场景
2. 掌握面向智能系统的轻量化网络设计技术（深度可分离卷积、分组卷积、复合缩放等）
3. 掌握面向部署的网络改进方案（量化、剪枝、知识蒸馏、图优化）
4. 理解深度学习模型开放标准 ONNX 的转换与优化流程
5. 掌握深度学习模型设计与训练的完整流程
6. 在昇腾 NPU 上运行代码，体验 AI 计算加速

---



## 4.1 深度学习网络架构体系

深度学习是人工智能领域的关键技术，其核心能力在于从数据中自动学习多层次的特征表示。面对不同的数据结构和任务需求，研究者开发了多种模型架构。深度学习通过构建多层次的非线性变换，能够从原始数据中自动学习有效的特征表示。不同的网络架构针对特定的数据模式和任务需求，形成了各自独特的设计理念和技术特点。

深度学习的核心在于**网络架构设计**——不同的架构擅长不同的任务。下面介绍三大主流架构。

### 4.1.1 常用深度学习网络介绍



#### 1. 卷积神经网络（CNN）：空间特征提取的专家

卷积神经网络是处理具有网格拓扑结构数据的经典架构，其核心设计理念源于对生物视觉系统的模拟，并在计算机视觉领域取得了卓越成就。

CNN 的三大核心设计原则：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">设计原则</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>局部连接</strong></td>
<td style="text-align: left;">每个神经元仅与输入的局部区域相连，大幅减少参数</td>
</tr>
<tr>
<td style="text-align: left;"><strong>权值共享</strong></td>
<td style="text-align: left;">卷积核在整个空间共享权重，赋予平移不变性</td>
</tr>
<tr>
<td style="text-align: left;"><strong>层次化提取</strong></td>
<td style="text-align: left;">浅层检测边缘→中层组合纹理→深层学习语义</td>
</tr>
</table>

**三大原则详解**：
- **局部连接**：与传统全连接网络不同，卷积核只覆盖输入的一个小区域（如 3×3），这意味着每个神经元只"看到"局部特征。以 3×3 卷积核为例，每个输出像素仅与 9 个输入像素相连，而非全部。这使得参数量从 $O(n^2)$ 降为 $O(k^2)$，极大减少了计算和存储开销。
- **权值共享**：同一个卷积核在整张特征图上滑动，所有位置共享同一组权重。这意味着无论目标出现在图像的哪个位置，都能被同一组权重检测到——这就是**平移不变性**。一个 3×3 卷积核只有 9 个参数，无论输入多大，参数量不变。
- **层次化提取**：网络越深，感受野越大，能学到的特征越抽象。第 1 层可能只检测边缘和颜色梯度，第 2~3 层将这些边缘组合成角点和纹理，更深层则能识别出物体部件乃至完整语义概念。这种由简到繁、由具体到抽象的特征学习过程，与人类视觉系统的认知机制高度吻合，是 CNN 在视觉任务上表现优异的根本原因。

**池化操作**在 CNN 中发挥着重要作用。最大池化是最常用的池化方式，它通过对局部区域取最大值来实现特征图的下采样。这种操作不仅逐步减小了特征图尺寸，降低了计算复杂度，还使特征对位置变化具有一定的鲁棒性，同时保留了最显著的特征响应。

**技术演进**：LeNet-5(1998) → AlexNet(2012) → VGGNet(2014) → ResNet(2015) → EfficientNet(2019)

> **演进脉络**：LeNet-5 首次将卷积+池化用于手写数字识别；AlexNet 引入 ReLU 激活和 Dropout，在 ImageNet 上取得突破；VGGNet 探索了更深的网络和更小的 3×3 卷积核；ResNet 通过残差连接解决了深层网络的梯度消失问题，使训练上百层的网络成为可能；EfficientNet 则通过复合缩放策略在精度和效率间取得最优平衡。

<img src="../../images/NN.png" alt="神经网络" width="500" style="display: block; margin-left: 0;" />



#### 动手实验：用 PyTorch 构建一个简单 CNN

下面我们在 NPU 上构建一个用于 MNIST 手写数字识别的 CNN，并完成训练与推理。

**代码说明**：
- 首先导入 `torch`、`torch.nn` 和 `torch_npu`（昇腾 NPU 适配层），并检测 NPU 是否可用。
- `SimpleCNN` 网络结构为：`Conv2d(1→32) → ReLU → MaxPool(2) → Conv2d(32→64) → ReLU → MaxPool(2) → Flatten → Linear(3136→10)`。
- 输入为 1×28×28 的灰度图，经过两次 2×2 池化后空间尺寸从 28→14→7，因此全连接层输入维度为 64×7×7=3136。
- 最后将模型迁移到 NPU 设备并打印模型结构和总参数量。

**预期结果**：
- `计算设备: npu`（如果 NPU 可用）
- `NPU 型号: Ascend910B3`（或类似型号）
- 打印出 `SimpleCNN` 的完整网络结构，包含 conv1、relu1、conv2、relu2、pool、fc 各层
- 总参数量约为 3.27M（主要来自全连接层 3136×10=31360 和两个卷积层）

> **为什么参数量主要来自全连接层？** 因为全连接层的每个输入节点都与所有输出节点相连，3136×10=31360 个权重，而卷积层通过权值共享，参数量仅为 kernel_size²×in_channels×out_channels，远小于全连接。



In [ ]:
!pip install onnxscript -q
import torch
import torch.nn as nn
import torch_npu  # 昇腾 NPU 适配层

device = torch.device('npu' if torch.npu.is_available() else 'cpu')
print(f'计算设备: {device}')
if torch.npu.is_available():
    print(f'NPU 型号: {torch.npu.get_device_name(0)}')

# 定义一个简单的 CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1, 1)   # 卷积: 1->32通道
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)  # 卷积: 32->64通道
        self.relu2 = nn.ReLU()
        self.pool = nn.MaxPool2d(2)               # 池化: 每次空间尺寸减半(28->14->7)
        self.fc = nn.Linear(64 * 7 * 7, 10)       # 全连接: 64通道×7×7=3136 ->10类

    def forward(self, x):
        x = self.pool(self.relu1(self.conv1(x)))
        x = self.pool(self.relu2(self.conv2(x)))
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'\n模型结构:\n{model}')
print(f'\n总参数量: {total_params:,}')


#### 2. 循环神经网络（RNN）：序列建模的经典方案

循环神经网络系列专门用于处理序列数据，其核心优势在于能够通过循环连接维护时序上的记忆能力，在自然语言处理、语音识别和时间序列分析等领域发挥着重要作用。

传统 RNN 按时间步顺序处理序列数据，通过隐藏状态传递历史信息。每个时间步的隐藏状态不仅取决于当前输入，还包含了之前所有时间步的上下文信息。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">变体</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;"><strong>LSTM</strong></td>
<td style="text-align: left;">三门控（遗忘/输入/输出），解决长期依赖</td>
</tr>
<tr>
<td style="text-align: left;"><strong>GRU</strong></td>
<td style="text-align: left;">LSTM 简化版，更新门+重置门，参数更少</td>
</tr>
<tr>
<td style="text-align: left;"><strong>BiRNN</strong></td>
<td style="text-align: left;">双向处理，包含过去和未来上下文</td>
</tr>
</table>

> **核心挑战**：链式结构导致梯度消失/爆炸，限制学习长期依赖。

**RNN 变体详解**：
- **LSTM（长短期记忆网络）**：引入了三个门控机制——**遗忘门**决定丢弃哪些历史信息，**输入门**决定写入哪些新信息，**输出门**决定输出哪些信息。通过门控机制，LSTM 能够选择性地保留或遗忘信息，有效缓解了标准 RNN 中的梯度消失问题，可以学习跨越数百个时间步的长期依赖。
- **GRU（门控循环单元）**：将 LSTM 的三个门简化为**更新门**和**重置门**两个门，合并了记忆单元和隐藏状态，参数量比 LSTM 少约 1/3，训练更快，在多数任务上性能与 LSTM 接近。
- **BiRNN（双向 RNN）**：由两个方向相反的 RNN 组成，一个从左到右处理序列，另一个从右到左。最终输出是两个方向隐藏状态的拼接。这使得模型在预测当前位置时能同时利用**过去和未来**的上下文信息，在 NLP 任务（如命名实体识别）中效果显著。

> **梯度消失/爆炸的原因**：RNN 在反向传播时需要将权重矩阵连乘 T 次（T 为序列长度），当权重矩阵的最大特征值大于 1 时梯度爆炸，小于 1 时梯度消失。LSTM 通过门控机制使梯度可以"绕过"连乘路径，从而缓解此问题。



#### 3. Transformer：基于自注意力的革命性架构

2017 年 Transformer 横空出世，完全依赖自注意力机制的设计理念，在自然语言处理领域引发了一场革命，并逐步扩展到计算机视觉等其他领域。其核心创新是**自注意力机制**：

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">全局感受野</td>
<td style="text-align: left;">一步关联任意距离的 token</td>
</tr>
<tr>
<td style="text-align: left;">高度并行</td>
<td style="text-align: left;">所有位置同时计算，充分利用 NPU 算力</td>
</tr>
<tr>
<td style="text-align: left;">可扩展性</td>
<td style="text-align: left;">堆叠更多层就能持续提升（Scaling Law）</td>
</tr>
</table>

Transformer 已成为大模型基础架构：GPT、BERT、Qwen、DeepSeek 等全部基于 Transformer。

**自注意力机制详解**：
- 公式中 $Q$（Query）、$K$（Key）、$V$（Value）分别由输入经过三个线性变换得到。$QK^\top$ 计算的是当前 token 与所有其他 token 的相似度（点积），除以 $\sqrt{d_k}$ 是为了防止点积值过大导致 softmax 梯度消失。
- softmax 将相似度归一化为注意力权重，再与 $V$ 加权求和，得到每个位置的输出。
- **全局感受野**：与 CNN 的局部卷积不同，自注意力一步就能关联序列中任意距离的两个 token，没有"距离衰减"问题。
- **高度并行**：RNN 必须按时间步串行计算，而自注意力的所有位置可以同时计算矩阵乘法，充分利用 NPU/GPU 的并行算力。
- **可扩展性**：实验发现，只要增加参数量和训练数据，Transformer 的性能就能持续提升（即 Scaling Law），这是大模型时代的基础。

> **ViT（Vision Transformer）** 成功将这一范式扩展至计算机视觉领域：通过将图像分割为固定大小的图块并添加位置编码，ViT 将图像处理转化为序列处理问题，证明了自注意力机制在纯视觉任务上同样可以达到领先性能。



#### 动手实验：体验自注意力机制

下面用 PyTorch 在 NPU 上实现一个简单的自注意力层，直观感受 Q、K、V 的计算。

**代码说明**：
- `SelfAttention` 类实现了多头自注意力：`embed_dim=64`（嵌入维度），`num_heads=4`（注意力头数），每个头的维度 `head_dim=16`。
- `qkv` 线性层一次性计算 Q、K、V（输出维度为 `embed_dim*3`），然后 reshape 分离。
- 注意力计算：`attn = softmax(Q·K^T / √d_k) · V`，最后经过 `proj` 线性层输出。
- 输入为 `(batch=4, seq_len=10, embed_dim=64)` 的随机张量。

**预期结果**：
- `输入形状: torch.Size([4, 10, 64])  (batch, seq_len, embed_dim)`
- `输出形状: torch.Size([4, 10, 64])  (batch, seq_len, embed_dim)` —— 输入输出形状相同，这是自注意力的特点
- `设备: npu:0` —— 运行在昇腾 NPU 上
- `自注意力机制在 NPU 上运行成功！`

> **为什么输入输出形状相同？** 自注意力是对序列中每个位置的特征进行重新加权组合，不改变序列长度和特征维度。每个输出位置是所有输入位置的加权平均，权重由注意力机制动态计算。



In [ ]:
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k3, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k3.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)

# 在 NPU 上运行自注意力
embed_dim, seq_len, batch = 64, 10, 4
attn_layer = SelfAttention(embed_dim).to(device)
x = torch.randn(batch, seq_len, embed_dim).to(device)
out = attn_layer(x)
print(f'输入形状: {x.shape}  (batch, seq_len, embed_dim)')
print(f'输出形状: {out.shape}  (batch, seq_len, embed_dim)')
print(f'设备: {out.device}')
print('自注意力机制在 NPU 上运行成功！')


---

### 4.1.2 面向智能系统常见轻量化深度学习网络

在深度学习技术快速发展的进程中，卷积神经网络和 Transformer 架构分别在计算机视觉和自然语言处理领域取得了突破性进展。然而，这些高性能模型普遍面临着**参数量庞大、计算复杂度高、内存占用多**等问题，严重制约了其在资源受限环境中的部署应用。为了推动人工智能技术在移动端、嵌入式设备和边缘计算等场景的落地，研究者们提出了一系列模型轻量化改进方法，通过结构创新和算法优化，在保持模型性能的同时显著提升推理效率。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">轻量化维度</th>
<th style="text-align: left;">核心思想</th>
<th style="text-align: left;">代表模型</th>
</tr>
<tr>
<td style="text-align: left;"><strong>结构重设计</strong></td>
<td style="text-align: left;">用更少的参数达到相近的表达能力</td>
<td style="text-align: left;">MobileNet、ShuffleNet、SqueezeNet</td>
</tr>
<tr>
<td style="text-align: left;"><strong>计算优化</strong></td>
<td style="text-align: left;">降低单次运算的计算复杂度</td>
<td style="text-align: left;">深度可分离卷积、分组卷积</td>
</tr>
<tr>
<td style="text-align: left;"><strong>智能搜索</strong></td>
<td style="text-align: left;">自动搜索最优网络结构</td>
<td style="text-align: left;">EfficientNet、FBNet</td>
</tr>
<tr>
<td style="text-align: left;"><strong>混合架构</strong></td>
<td style="text-align: left;">融合 CNN 与 Transformer 优势</td>
<td style="text-align: left;">MobileViT、Mobile-Former</td>
</tr>
</table>

<img src="../../images/training_inference.png" alt="训练与推理" width="500" style="display: block; margin-left: 0;" />



#### 1. 卷积神经网络轻量化技术

卷积神经网络轻量化的核心目标是在**尽可能保持模型精度的前提下，大幅减少参数数量和计算量**。这一技术路线主要通过结构重设计和计算优化两个维度实现。

##### 深度可分离卷积（MobileNet 系列）

深度可分离卷积技术是卷积神经网络轻量化的**里程碑式突破**。该技术将标准卷积分解为两个独立的操作：

- **深度卷积（Depthwise Conv）**：负责在单个输入通道上进行空间特征提取，每个通道独占一个卷积核；
- **逐点卷积（Pointwise Conv）**：用 1×1 卷积负责跨通道的特征融合。

这种分解策略使得计算复杂度从标准卷积的 $O(K^2 \cdot C_{in} \cdot C_{out})$ 降低到 $O(K^2 \cdot C_{in} + C_{in} \cdot C_{out})$，其中 $K$ 为卷积核尺寸，$C_{in}$ 和 $C_{out}$ 分别为输入和输出通道数。

> **直观理解**：标准卷积同时做"空间滤波"和"通道融合"两件事；深度可分离卷积把这两件事拆开分别做，先逐通道滤波（参数极少），再用 1×1 卷积融合通道。当 $C_{out}$ 较大时，计算量可降低到原来的 1/8 ~ 1/9。

**MobileNet 系列模型**将这一技术发挥到极致：
- **MobileNetV1**：首次系统使用深度可分离卷积构建轻量网络；
- **MobileNetV2**：引入**倒残差结构**（Inverted Residual）和**线性瓶颈层**（Linear Bottleneck），先升维再降维，优化信息流动；
- **MobileNetV3**：结合 **SE 注意力机制**实现通道间的自适应权重调整，最终在 ImageNet 数据集上以仅 **400 万参数**的规模达到 **70% 以上的 top-1 准确率**。



##### 分组卷积与通道混洗（ShuffleNet）

分组卷积与通道混洗机制为轻量化设计提供了另一条有效路径。

- **分组卷积**：将输入通道划分为 $g$ 个组，在每个组内独立进行卷积运算，显著减少了跨通道的计算开销，计算量降低到传统卷积的 $1/g$。
- **通道混洗（Channel Shuffle）**：分组后不同组之间特征不流通，ShuffleNet 创新性地引入通道混洗操作，通过有规律的通道重排促进组间信息交换。

> **为什么需要通道混洗？** 如果只做分组卷积，组与组之间永远不交流，特征表达受限。通道混洗在每次分组卷积后重排通道顺序，使下一层分组卷积能跨组获取信息，兼顾效率与表达力。

这种设计使得模型在保持较强特征表达能力的同时，将计算量降低到传统卷积的 $1/g$（$g$ 为分组数），特别适合在计算资源极其有限的嵌入式视觉系统中部署。

##### 复合缩放与神经架构搜索（EfficientNet / FBNet）

复合缩放与神经架构搜索代表了轻量化技术的**智能化发展方向**。

- **EfficientNet 复合缩放**：系统性地平衡网络的**深度** $d$、**宽度** $w$ 和**输入分辨率** $r$ 三个维度，通过简单的复合系数 $\phi$ 统一控制：
$$d = \alpha^\phi, \quad w = \beta^\phi, \quad r = \gamma^\phi, \quad \text{s.t.} \quad \alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$$
  这种方法在理论上保证了模型容量与计算资源的优化匹配，在实践中实现了从 EfficientNet-B0 到 B7 的平滑扩展。
- **FBNet 硬件感知 NAS**：采用可微分架构搜索（DARTS），将**目标硬件的实际延迟**纳入优化目标，自动发现针对特定平台的最优网络结构，实现模型精度与推理速度的帕累托最优。

> **复合缩放的直觉**：单独加深、加宽或提高分辨率都会遇到收益递减。同时按比例缩放三个维度，就像"均衡营养"——每一维都恰到好处，用同样的计算量获得更高精度。



##### 极限参数压缩（SqueezeNet / ESPNet）

极限参数压缩技术专注于解决**极端资源约束**下的模型部署问题。

- **SqueezeNet**：通过精心设计的 **Fire 模块**，使用大量的 1×1 卷积核进行通道压缩和扩展，将 3×3 卷积的比例控制在合理范围内，最终以仅 **0.5MB** 的模型大小实现了与 AlexNet 相当的精度（AlexNet 约 240MB）。
- **ESPNet**：通过高效空间金字塔模块（EESP），结合多尺度空洞卷积和分组卷积，在保持较大感受野的同时严格控制参数增长，为实时语义分割任务提供了高效的解决方案。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">参数量/大小</th>
<th style="text-align: left;">关键技术</th>
<th style="text-align: left;">目标任务</th>
</tr>
<tr>
<td style="text-align: left;">MobileNetV3</td>
<td style="text-align: left;">~4M</td>
<td style="text-align: left;">深度可分离卷积+SE+倒残差</td>
<td style="text-align: left;">通用图像分类</td>
</tr>
<tr>
<td style="text-align: left;">ShuffleNetV2</td>
<td style="text-align: left;">~2M</td>
<td style="text-align: left;">分组卷积+通道混洗</td>
<td style="text-align: left;">嵌入式视觉</td>
</tr>
<tr>
<td style="text-align: left;">EfficientNet-B0</td>
<td style="text-align: left;">~5M</td>
<td style="text-align: left;">复合缩放</td>
<td style="text-align: left;">高效分类</td>
</tr>
<tr>
<td style="text-align: left;">SqueezeNet</td>
<td style="text-align: left;">0.5MB</td>
<td style="text-align: left;">Fire模块+1×1卷积</td>
<td style="text-align: left;">极限压缩</td>
</tr>
<tr>
<td style="text-align: left;">ESPNet</td>
<td style="text-align: left;">~1M</td>
<td style="text-align: left;">空洞金字塔+分组卷积</td>
<td style="text-align: left;">实时语义分割</td>
</tr>
</table>



##### 动手实验：体验深度可分离卷积的参数压缩

下面用 PyTorch 对比标准卷积与深度可分离卷积的参数量和计算量，直观感受轻量化的效果。

**代码说明**：
- `StandardConv`：一个普通的 `Conv2d(32→64, 3×3)`，参数量为 $3 \times 3 \times 32 \times 64 = 18432$。
- `DepthwiseSeparableConv`：先 `Depthwise`（`Conv2d(32→32, 3×3, groups=32)`，参数量 $3 \times 3 \times 32 = 288$），再 `Pointwise`（`Conv2d(32→64, 1×1)`，参数量 $32 \times 64 = 2048$），合计 2336。
- 对比两者的参数量比值，验证深度可分离卷积的压缩效果。

**预期结果**：
- `标准卷积参数量: 18,432`
- `深度可分离卷积参数量: 2,336`（288 + 2048）
- `参数压缩比: 约 7.9x`（18432 / 2336 ≈ 7.89）
- `深度可分离卷积参数量显著减少！`

> **压缩比为何约为 1/N_out + 1/K²？** 理论上，深度可分离卷积参数量 / 标准卷积参数量 = 1/N_out + 1/K² = 1/64 + 1/9 ≈ 0.127，即压缩约 7.9 倍。



In [ ]:
# 对比标准卷积与深度可分离卷积的参数量
class StandardConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, 1, 1)
    def forward(self, x):
        return self.conv(x)

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.depthwise = nn.Conv2d(cin, cin, 3, 1, 1, groups=cin)  # 逐通道卷积
        self.pointwise = nn.Conv2d(cin, cout, 1)                    # 1x1 跨通道融合
    def forward(self, x):
        return self.pointwise(self.depthwise(x))

cin, cout = 32, 64
std_conv = StandardConv(cin, cout).to(device)
ds_conv  = DepthwiseSeparableConv(cin, cout).to(device)

p_std = sum(p.numel() for p in std_conv.parameters())
p_ds  = sum(p.numel() for p in ds_conv.parameters())

print(f'标准卷积参数量: {p_std:,}')
print(f'深度可分离卷积参数量: {p_ds:,}')
print(f'参数压缩比: {p_std / p_ds:.1f}x')
print('深度可分离卷积参数量显著减少！')


#### 2. Transformer 模型轻量化技术

Transformer 模型的轻量化主要针对**自注意力机制的计算复杂度和内存占用**问题进行优化。标准自注意力的计算复杂度为 $O(N^2)$（$N$ 为序列长度），在处理长序列或部署到移动端时成为瓶颈。通过结构简化和混合架构设计可实现效率提升。

##### 混合架构设计（MobileViT）

混合架构设计是轻量化 Transformer 的重要方向。**MobileViT** 创造性地将小规模 Transformer 模块嵌入到轻量级 CNN 架构中：

- 利用**卷积层**进行局部特征提取（高效、参数少）；
- 仅对**高级特征块**应用 Transformer 处理（全局建模）。

这种设计既保留了 CNN 的高效局部建模能力，又引入了 Transformer 的全局依赖捕获优势，同时将自注意力的计算复杂度从 $O(N^2)$ 降低到 $O(M^2)$，其中 $M$ 远小于原始序列长度 $N$。

> **效果**：MobileViT 在 ImageNet 数据集上以 **600 万参数**的规模达到 **78% 的准确率**，推理速度在移动设备上达到每秒数十帧。

##### 并行双分支架构（Mobile-Former）

并行双分支架构为特征提取提供了新的思路。**Mobile-Former** 采用 Mobile 分支和 Former 分支并行协作的设计：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">分支</th>
<th style="text-align: left;">架构</th>
<th style="text-align: left;">职责</th>
</tr>
<tr>
<td style="text-align: left;"><strong>Mobile 分支</strong></td>
<td style="text-align: left;">CNN</td>
<td style="text-align: left;">处理高分辨率局部特征</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Former 分支</strong></td>
<td style="text-align: left;">Transformer</td>
<td style="text-align: left;">专注于全局上下文建模</td>
</tr>
</table>

两个分支通过**双向交叉注意力桥**实现实时信息交互：局部特征通过全局上下文得到增强，全局表示通过局部细节得到补充。这种设计使得模型能够以极低的计算开销同时捕获多尺度特征，在 COCO 目标检测任务上以仅 **1/3 的计算量**达到与对应 CNN 模型相当的精度。



##### 纯 Transformer 结构优化（EfficientFormer）

纯 Transformer 结构优化致力于从根本上改进标准 Transformer 的效率问题。**EfficientFormer** 通过详细的延迟分析发现，标准 Transformer 中某些操作在移动设备上的**实际延迟与理论 FLOPs 并不匹配**。

基于这一洞察，该模型采用以下优化策略：
- 采用**池化前馈网络**作为主要特征混合器，仅在关键位置使用经过优化的自注意力模块；
- 在整个前向过程中保持 **4D 张量表示**，避免了频繁的 reshape 操作带来的开销。

> **效果**：EfficientFormer 在移动 GPU 上的推理速度比同类模型快 **3 倍以上**，同时保持有竞争力的准确率。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">参数量</th>
<th style="text-align: left;">关键创新</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">MobileViT</td>
<td style="text-align: left;">~6M</td>
<td style="text-align: left;">CNN+Transformer 混合</td>
<td style="text-align: left;">移动端视觉</td>
</tr>
<tr>
<td style="text-align: left;">Mobile-Former</td>
<td style="text-align: left;">-</td>
<td style="text-align: left;">并行双分支+交叉注意力</td>
<td style="text-align: left;">目标检测</td>
</tr>
<tr>
<td style="text-align: left;">EfficientFormer</td>
<td style="text-align: left;">-</td>
<td style="text-align: left;">池化FFN+4D张量</td>
<td style="text-align: left;">移动GPU推理</td>
</tr>
</table>



#### 3. 轻量化技术的通用原则与发展趋势

尽管各类轻量化方法在具体实现上存在差异，但它们都遵循一些**共同的设计原则**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">原则</th>
<th style="text-align: left;">含义</th>
<th style="text-align: left;">典型应用</th>
</tr>
<tr>
<td style="text-align: left;"><strong>计算分解</strong></td>
<td style="text-align: left;">将复杂运算拆分为多个简单操作的组合</td>
<td style="text-align: left;">深度可分离卷积将空间滤波与通道融合分离</td>
</tr>
<tr>
<td style="text-align: left;"><strong>特征重用</strong></td>
<td style="text-align: left;">通过残差/密集连接促进信息流动，提高参数利用率</td>
<td style="text-align: left;">ResNet 残差连接、DenseNet 密集连接</td>
</tr>
<tr>
<td style="text-align: left;"><strong>硬件感知</strong></td>
<td style="text-align: left;">将目标平台特性纳入设计考虑</td>
<td style="text-align: left;">FBNet 延迟感知 NAS</td>
</tr>
</table>

当前轻量化技术的发展呈现出**三个明显趋势**：

1. **自动化设计的普及**：神经架构搜索（NAS）和自动化机器学习（AutoML）技术正在成为轻量化模型设计的主流方法，减少人工调参的依赖。
2. **跨平台适配能力的增强**：新一代轻量化模型能够更好地适应从 CPU、GPU 到各种专用 AI 芯片的不同硬件平台。
3. **多模态统一架构的出现**：统一的轻量化 backbone 网络开始支持视觉、语言等多种模态的处理。

> **总结**：这些技术进步极大地推动了深度学习在现实场景中的应用，使得复杂的 AI 模型能够在智能手机、物联网设备、自动驾驶系统等资源受限环境中高效运行。随着边缘计算需求的持续增长，模型轻量化技术将继续向着**更高效、更智能、更通用**的方向发展。



---

### 4.1.3 面向智能系统部署的网络改进方案

面向智能系统的深度学习模型改进是一个**系统工程**，需要从多个维度协同优化。智能设备存在计算能力有限、存储空间紧张、功耗预算严格等约束条件，这就要求深度学习模型必须经过深度优化才能实现有效部署。

当前主流的优化方法形成了完整的优化链条，涵盖**量化、剪枝、知识蒸馏和图优化**等多个技术方向：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方法</th>
<th style="text-align: left;">优化层面</th>
<th style="text-align: left;">核心作用</th>
</tr>
<tr>
<td style="text-align: left;"><strong>量化</strong></td>
<td style="text-align: left;">数值精度</td>
<td style="text-align: left;">降低数值精度，减少存储和计算开销</td>
</tr>
<tr>
<td style="text-align: left;"><strong>剪枝</strong></td>
<td style="text-align: left;">模型结构</td>
<td style="text-align: left;">移除冗余结构，简化模型复杂度</td>
</tr>
<tr>
<td style="text-align: left;"><strong>知识蒸馏</strong></td>
<td style="text-align: left;">知识迁移</td>
<td style="text-align: left;">通过大模型指导小模型实现压缩</td>
</tr>
<tr>
<td style="text-align: left;"><strong>图优化</strong></td>
<td style="text-align: left;">执行效率</td>
<td style="text-align: left;">计算图重构，提升执行效率</td>
</tr>
</table>

这些技术手段相互补充、协同作用，构成了面向嵌入式系统的深度学习模型优化体系。



#### 1. 深度学习量化方法

##### （1）量化理论基础与实现机制

深度学习量化是一种通过**降低数值表示精度**来实现模型压缩和加速的关键技术。其数学本质是将神经网络中的权重和激活值从高精度浮点数表示转换为低精度数值表示，这个过程涉及复杂的数值映射和误差控制。

在 32 位浮点数到 8 位整数的转换过程中，**量化误差**主要来源于两个方面：
- **舍入误差**：由于浮点数值需要映射到有限的整数区间而产生的精度损失；
- **截断误差**：当数值范围超出目标整数表示范围时产生的信息丢失。

为了最小化这些误差，研究者开发了多种量化策略：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">策略</th>
<th style="text-align: left;">特点</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;"><strong>对称量化</strong></td>
<td style="text-align: left;">以零为中心对称分布，计算简单</td>
<td style="text-align: left;">权重分布近似对称时</td>
</tr>
<tr>
<td style="text-align: left;"><strong>非对称量化</strong></td>
<td style="text-align: left;">引入零点偏移量，适应非对称分布</td>
<td style="text-align: left;">激活值分布偏斜时</td>
</tr>
<tr>
<td style="text-align: left;"><strong>动态量化</strong></td>
<td style="text-align: left;">推理时动态计算激活值量化参数</td>
<td style="text-align: left;">动态范围变化大的模型</td>
</tr>
<tr>
<td style="text-align: left;"><strong>静态量化</strong></td>
<td style="text-align: left;">校准阶段确定量化参数，推理无额外开销</td>
<td style="text-align: left;">部署推理效率优先</td>
</tr>
<tr>
<td style="text-align: left;"><strong>混合精度</strong></td>
<td style="text-align: left;">敏感层高精度，冗余层低精度</td>
<td style="text-align: left;">精度与效率最佳平衡</td>
</tr>
</table>

> **混合精度量化的直觉**：并非所有层对量化的敏感度相同。输出层对精度敏感，保持 FP32；中间卷积层计算密集，用 INT8 加速。这样在整体加速的同时保护关键精度。

##### （2）量化实施流程与技术细节

量化过程的具体实施包含**校准**和**转换**两个关键阶段：

- **校准阶段**：收集代表性的输入数据，通过统计分析确定权重和激活值的动态范围。常用校准方法：
  - **最大最小值法**：直接使用统计到的极值作为范围边界，实现简单但容易受异常值影响；
  - **KL 散度法**：通过最小化原始分布与量化分布的差异来确定最优范围，精度更高但计算复杂；
  - **移动平均法**：对动态范围做滑动平均，稳定性更好。
- **转换阶段**：基于校准得到的量化参数，将浮点运算转换为整数运算。例如在卷积运算中，需要将浮点卷积分解为**整数卷积 + 尺度调整 + 偏置校正**。

> 现代深度学习框架如 **TensorFlow Lite** 和 **PyTorch Mobile** 提供了完整的量化工具链，支持从模型准备、校准到最终转换的全流程自动化处理。

##### （3）量化技术演进与前沿发展

当前量化技术正朝着**极低精度**方向发展：
- **二值化/三值化网络**：将权重压缩到 1~2 比特，实现极高的压缩比和能效提升；
- **自适应量化**：根据输入数据特性动态调整量化参数，提升真实场景中的鲁棒性；
- **硬件感知量化**：充分考虑目标硬件的特性，针对不同处理器架构优化量化策略。



#### 2. 深度学习剪枝方法

##### （1）剪枝理论基础与分类体系

深度学习剪枝是一种通过**移除神经网络中的冗余参数**来压缩模型规模的方法。其理论依据在于深度学习模型普遍存在**过度参数化**现象——大量参数对最终输出的贡献微乎其微。

根据剪枝粒度的不同，可以分为两大类：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">类型</th>
<th style="text-align: left;">操作单元</th>
<th style="text-align: left;">压缩率</th>
<th style="text-align: left;">硬件加速</th>
</tr>
<tr>
<td style="text-align: left;"><strong>非结构化剪枝</strong></td>
<td style="text-align: left;">单个权重</td>
<td style="text-align: left;">高</td>
<td style="text-align: left;">难（需稀疏存储支持）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>结构化剪枝</strong></td>
<td style="text-align: left;">通道/滤波器</td>
<td style="text-align: left;">较低</td>
<td style="text-align: left;">易（直接改变网络结构）</td>
</tr>
</table>

> **初学者提示**：非结构化剪枝像"把树上的个别叶子摘掉"——树还是那么大，只是叶子稀疏了，普通硬件不会变快。结构化剪枝像"把整根树枝砍掉"——树真的变小了，硬件能直接加速。

##### （2）剪枝算法设计与实现

有效的剪枝算法需要解决三个核心问题：

1. **重要性评估准则**——判断哪些参数可以剪：
   - **权重幅值准则**：假设小权重对输出影响小，实现简单但精度有限；
   - **梯度信息准则**：通过反向传播梯度评估参数重要性，更能反映对损失函数的影响；
   - **Hessian 矩阵准则**：从二阶优化角度评估，精度最高但计算复杂度最大。
2. **剪枝策略**——如何剪：
   - **一次性剪枝**：直接按目标稀疏度剪枝后微调，大比例剪枝时容易性能崩溃；
   - **迭代剪枝**：通过"剪枝→微调"的多次循环逐步达到目标稀疏度，更好地保持性能。
3. **恢复优化方法**——剪后如何恢复精度：通常通过微调（fine-tuning）在训练集上继续训练若干 epoch。

##### （3）先进剪枝技术与优化策略

近年来剪枝技术呈现多个发展方向：
- **自动剪枝**：通过 NAS 自动确定各层最佳稀疏度，避免手动调参；
- **动态剪枝**：根据输入样本特性动态调整网络结构，实现自适应计算分配；
- **联合优化**：将剪枝与其他压缩技术结合，如**量化感知剪枝**、**蒸馏引导剪枝**等，实现多重压缩叠加。

> 现代剪枝方法能在保持模型性能的同时，实现**一个数量级以上**的压缩比。



#### 3. 深度学习蒸馏方法

##### （1）知识蒸馏的理论基础

知识蒸馏的核心思想是通过**"教师-学生"框架**实现知识迁移。其理论依据在于：深度神经网络不仅学习输入到输出的映射函数，还在训练过程中捕获了数据中丰富的**结构化信息**。这些信息体现在模型输出的概率分布中，包含了类别间的相似性关系、数据的内在结构等无法从硬标签中获得的**暗知识**。

**温度参数** $T$ 的引入是知识蒸馏的关键创新：

$$\text{soft\_label}_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

较高的温度值 $T$ 产生更平滑的概率分布，能够凸显类别间的细微差异，使学生模型更好地学习教师模型的泛化特性。

> **暗知识举例**：对于一张猫的图片，硬标签只说"这是猫"。但教师的软标签可能说"80% 猫，15% 狗，3% 兔，2% 其他"——这些额外的相似性信息就是"暗知识"，能帮助学生模型学到更丰富的表征。

##### （2）蒸馏算法设计与优化

知识蒸馏的算法设计涉及多个关键要素：

- **知识表示形式**：不仅包括最终输出的软标签，还可扩展到：
  - **特征基蒸馏**：匹配教师和学生模型中间层的特征激活；
  - **注意力蒸馏**：对齐空间注意力图，提升特征选择能力；
  - **关系蒸馏**：保持样本间相似性关系，传递结构化知识。
- **损失函数设计**：需平衡硬标签的交叉熵损失和软标签的蒸馏损失：
$$L = \alpha \cdot L_{\text{hard}}(y, y_s) + (1-\alpha) \cdot T^2 \cdot L_{\text{soft}}(p_T, p_S)$$
  权重系数通常采用动态调整策略，训练初期侧重蒸馏损失，后期增加硬标签权重。

##### （3）蒸馏技术应用与发展趋势

知识蒸馏技术在嵌入式深度学习中的应用呈现多样化趋势：
- **在线蒸馏**：将蒸馏过程整合到模型训练中，实现端到端优化；
- **自蒸馏**：模型自己作为教师，通过特殊结构设计实现知识复用；
- **跨模态蒸馏**：将不同模态间的知识进行迁移，拓展应用范围。

> 知识蒸馏已成为提升轻量模型性能的重要手段，在移动端视觉识别、边缘智能推理等场景中发挥着重要作用。



#### 4. 深度学习图优化方法

##### （1）计算图优化理论基础

深度学习图优化建立在**计算图分析和变换**的理论基础之上。计算图作为深度学习模型的中间表示，捕获了所有操作之间的依赖关系和数据流。图优化技术通过对计算图进行**语义保持的变换**，改善计算顺序、减少冗余操作、优化内存访问模式，从而在不改变模型数学功能的前提下提升执行效率。

##### （2）优化技术分类与实现

图优化技术可以分为**局部优化**和**全局优化**两个层次：

**局部优化**——针对特定操作模式进行改进：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优化技术</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">效果</th>
</tr>
<tr>
<td style="text-align: left;"><strong>算子融合</strong></td>
<td style="text-align: left;">将多个连续操作合并为单一内核</td>
<td style="text-align: left;">减少内存访问开销和内核启动开销</td>
</tr>
<tr>
<td style="text-align: left;"><strong>常量传播</strong></td>
<td style="text-align: left;">编译期预计算常量表达式</td>
<td style="text-align: left;">减少运行时计算</td>
</tr>
<tr>
<td style="text-align: left;"><strong>公共子表达式消除</strong></td>
<td style="text-align: left;">避免重复计算</td>
<td style="text-align: left;">减少冗余计算</td>
</tr>
</table>

> **算子融合示例**：将 `Conv → BatchNorm → ReLU` 三个独立算子融合为一个复合内核，不仅减少了 2 次中间结果的写回和读取，还减少了 2 次内核启动开销。现代编译器如 **TVM**、**TensorRT** 能自动发现可融合的操作序列。

**全局优化**——从整个计算图角度进行结构性改进：
- **计算调度优化**：重排操作执行顺序，最大化硬件利用率；
- **内存规划**：通过张量生命周期分析实现内存复用，降低峰值内存需求；
- **并行化策略**：根据硬件特性分配计算任务，充分发挥多核性能。

##### （3）硬件感知优化与编译技术

现代图优化技术越来越注重硬件特性，发展出**硬件感知**的优化方法。这些方法针对特定处理器架构的特征（如内存层次结构、并行计算单元、特殊指令集等）进行定制化优化。深度学习编译器通过多层中间表示实现硬件无关优化与硬件特定优化的分离。

> **即时编译（JIT）**和自适应优化技术能够根据运行时信息动态调整优化策略，实现更好的性能表现。图优化成为深度学习部署中提升效率的关键环节，特别是在资源受限的嵌入式环境中发挥着不可替代的作用。



#### 5. 技术融合与系统优化

##### （1）多技术协同优化策略

在实际的嵌入式深度学习部署中，单一优化技术往往难以满足严格的约束要求，需要多种技术的**协同应用**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">组合方式</th>
<th style="text-align: left;">协同效应</th>
</tr>
<tr>
<td style="text-align: left;">量化 + 剪枝</td>
<td style="text-align: left;">同时减少参数数量和数值精度，实现叠加压缩</td>
</tr>
<tr>
<td style="text-align: left;">蒸馏 + 剪枝</td>
<td style="text-align: left;">教师模型指导剪枝后模型的恢复，提升恢复质量</td>
</tr>
<tr>
<td style="text-align: left;">图优化 + 其他</td>
<td style="text-align: left;">确保优化后模型在目标硬件上获得最佳性能</td>
</tr>
</table>

> **典型优化链**：先蒸馏得到小模型 → 再剪枝去除冗余权重 → 再量化为 INT8 → 最后图优化融合算子。每一步都在前一步的基础上进一步压缩，实现最大程度的轻量化。

##### （2）端到端优化流程设计

有效的模型优化需要设计系统化的端到端流程：

1. **模型分析**：剖析计算负载、内存访问模式等特性，确定优化重点；
2. **优化计划**：根据目标硬件约束和性能要求，制定各技术应用的顺序和参数；
3. **优化实施**：按计划逐步执行，建立评估体系监控各项指标变化；
4. **效果验证**：使用代表性数据集和真实部署场景验证优化效果。

##### （3）优化效果评估与验证

优化效果的评估需要综合考虑多个维度：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">评估维度</th>
<th style="text-align: left;">指标</th>
</tr>
<tr>
<td style="text-align: left;">模型精度</td>
<td style="text-align: left;">准确率、F1 等</td>
</tr>
<tr>
<td style="text-align: left;">推理速度</td>
<td style="text-align: left;">延迟、吞吐量</td>
</tr>
<tr>
<td style="text-align: left;">内存占用</td>
<td style="text-align: left;">峰值内存、模型体积</td>
</tr>
<tr>
<td style="text-align: left;">能耗效率</td>
<td style="text-align: left;">每焦耳推理次数</td>
</tr>
</table>

> 通过系统化的技术融合和流程优化，面向嵌入式系统的深度学习模型改进能够实现在严格资源约束下的高效部署。随着技术的不断发展，嵌入式深度学习优化将继续向着**自动化、智能化和自适应化**的方向演进。



---

### 4.1.4 深度学习模型的开放标准 ONNX

在当今人工智能技术快速演进的环境中，深度学习框架的多样性既促进了技术创新，也带来了严重的**模型互操作性问题**。TensorFlow、PyTorch、MXNet 等主流框架各自拥有独特的模型定义方式、算子实现和运行时环境，这种碎片化现状极大地阻碍了模型的跨平台部署和复用。

为了解决这一关键挑战，微软与 Facebook 等公司于 2017 年联合推出了 **ONNX（Open Neural Network Exchange）** 这一开放标准。它通过定义统一的模型表示规范和算子集，建立了连接不同训练框架与推理引擎的通用桥梁。

> **ONNX 的核心价值**：提供一种与框架无关的模型中间表示，使得开发者能够在任一主流框架中完成模型训练后，将其导出为标准化的 ONNX 格式，然后在各种支持 ONNX 的推理环境中无缝部署——实现"**一次训练，多处部署**"。

**ONNX 架构设计**体现了对兼容性、扩展性和性能的全面考量：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方面</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>标准化</strong></td>
<td style="text-align: left;">完整的前向计算图定义，涵盖卷积、池化、全连接、归一化等核心操作</td>
</tr>
<tr>
<td style="text-align: left;"><strong>扩展性</strong></td>
<td style="text-align: left;">支持用户自定义算子，适应新型网络结构和专用硬件加速需求</td>
</tr>
<tr>
<td style="text-align: left;"><strong>技术实现</strong></td>
<td style="text-align: left;">基于 Protocol Buffers 序列化，文件体积小、解析速度快、前后兼容</td>
</tr>
<tr>
<td style="text-align: left;"><strong>生态系统</strong></td>
<td style="text-align: left;">NVIDIA、Intel、AMD 等硬件厂商和众多软件框架广泛支持</td>
</tr>
</table>



#### 1. ONNX 模型转换的详细流程与技术实现

ONNX 模型转换是整个生态中的核心环节，其本质是将特定深度学习框架中训练完成的模型导出为 ONNX 格式，或者将 ONNX 模型进一步转换为目标推理引擎或硬件支持的专用格式。

模型转换的具体流程包含**三个关键阶段**：

##### 阶段一：训练模型导出

不同框架提供了相应的导出接口。以 PyTorch 为例，开发者可以使用 `torch.onnx.export()` 函数将训练好的模型转换为 ONNX 格式。这个过程中需要特别关注：
- **输入张量的形状定义**：对于包含动态维度的模型（如处理可变长度序列的 NLP 模型），需要明确指定**动态轴**（dynamic_axes）参数；
- **算子集版本**：不同版本的 ONNX 标准可能对算子语义有细微差异，需指定 `opset_version`。

在 TensorFlow 生态中，可以通过 **tf2onnx** 工具实现模型转换，自动完成图结构的映射和优化。

##### 阶段二：模型校验

ONNX 提供了 `onnx.checker` 工具来验证模型的合规性，检查模型的图结构、节点连接、算子版本兼容性等多个方面。对于复杂的模型，还可以使用 **ONNX Runtime** 进行推理验证，通过对比原始框架和 ONNX Runtime 的输出结果，确保数值计算的一致性。

##### 阶段三：模型优化

ONNX Optimizer 提供了一系列图级优化 pass：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优化 pass</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>算子融合</strong></td>
<td style="text-align: left;">如 Conv-BN-ReLU 的融合</td>
</tr>
<tr>
<td style="text-align: left;"><strong>常量折叠</strong></td>
<td style="text-align: left;">预计算图中的常量表达式</td>
</tr>
<tr>
<td style="text-align: left;"><strong>死代码消除</strong></td>
<td style="text-align: left;">移除不可达的计算节点</td>
</tr>
</table>

> 对于特定硬件平台，还可以使用供应商提供的专用优化工具进行进一步调优，例如 NVIDIA 的 **TensorRT** 或 Intel 的 **OpenVINO** 都能够接受 ONNX 模型作为输入。



#### 2. ONNX 模型优化的系统化方法

ONNX 模型优化是一个多层次、系统化的工程过程，涵盖了从算法层面的模型压缩到系统层面的计算图优化等多个维度。

##### 模型压缩技术层面

- **量化**：ONNX 支持**静态量化**和**动态量化**两种主要模式。
  - 静态量化：使用校准数据集确定激活值动态范围，同时量化权重和激活值，推理时无额外开销；
  - 动态量化：只量化权重，激活值在推理过程中动态量化，适合动态范围变化大的模型。
  - 支持灵活配置：对敏感的输出层保持 FP32，对计算密集的卷积层进行 INT8。
- **模型剪枝**：结构化剪枝（如通道剪枝）直接改变图结构，输出稠密计算图，受益于标准推理优化；非结构化剪枝压缩率更高，但需专门的稀疏推理运行时支持。
- **知识蒸馏**：ONNX 格式为师生模型的联合优化提供便利框架，可利用图组合功能实现特征对齐、注意力转移等高级蒸馏技术。

##### 计算图优化层面

ONNX Runtime 和相关优化工具提供了丰富的优化 pass：

- **常量折叠**：识别可预先计算的子表达式，替换为计算结果节点；
- **算子融合**：将相邻算子合并为复合算子（如 Conv+BatchNorm+ReLU → 单内核），减少内核启动开销和中间结果存储；
- **内存优化**：通过分析张量生命周期实现内存复用和高效调度，显著降低峰值内存需求。

> **总结**：ONNX 通过统一的中间表示打通了训练框架与推理引擎的壁垒，配合量化、剪枝、蒸馏和图优化等技术，为深度学习模型的高效跨平台部署提供了完整的工具链支持。



##### 动手实验：将 PyTorch 模型导出为 ONNX 格式

下面将前面定义的 `SimpleCNN` 导出为 ONNX 格式，体验"一次训练，多处部署"的流程。

**代码说明**：
- 将模型切换到 `eval()` 模式并放到 CPU 上（ONNX 导出通常在 CPU 上进行）。
- 创建一个虚拟输入 `dummy_input`，形状为 `(1, 1, 28, 28)`。
- 调用 `torch.onnx.export()` 导出模型，指定 `opset_version=11`，输入输出动态 batch 维。
- 尝试用 `onnx` 库校验模型，并用 `onnxruntime` 推理验证结果一致性。
- 若 `onnx` / `onnxruntime` 未安装，则跳过校验，仅完成导出。

**预期结果**：
- `ONNX 模型已导出: simple_cnn.onnx`
- 若安装了 onnx：`ONNX 模型校验通过！`
- 若安装了 onnxruntime：`PyTorch 输出与 ONNX Runtime 输出一致: True`

> **动态轴的意义**：指定 `dynamic_axes={'input': {0: 'batch'}}` 使得导出的 ONNX 模型可以接受任意 batch 大小的输入，在推理时更加灵活。



In [ ]:
# 将 SimpleCNN 导出为 ONNX 格式
import os
import numpy as np

onnx_path = 'simple_cnn.onnx'
model_cpu = SimpleCNN().eval()
dummy_input = torch.randn(1, 1, 28, 28)

torch.onnx.export(
    model_cpu, dummy_input, onnx_path,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=11
)
print(f'ONNX 模型已导出: {onnx_path}  (大小: {os.path.getsize(onnx_path)/1024:.1f} KB)')

# 尝试校验与推理验证
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print('ONNX 模型校验通过！')

    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)
    test_input = torch.randn(4, 1, 28, 28)
    with torch.no_grad():
        pt_out = model_cpu(test_input).numpy()
    ort_out = sess.run(None, {'input': test_input.numpy()})[0]
    consistent = np.allclose(pt_out, ort_out, atol=1e-4)
    print(f'PyTorch 输出与 ONNX Runtime 输出一致: {consistent}')
except ImportError:
    print('未安装 onnx / onnxruntime，已跳过校验（可 pip install onnx onnxruntime 后重试）')


---

## 4.2 深度学习模型设计与实现

### 4.2.1 主流深度学习框架对比

深度学习框架作为人工智能领域的核心基础设施，为神经网络模型的设计、训练与部署提供了高效、灵活且可扩展的工具平台。这些框架通过提供高级编程接口（主要基于 Python），对底层计算细节进行了深度封装，使得研究人员和工程师能够专注于算法逻辑和模型创新。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">框架</th>
<th style="text-align: left;">开发者</th>
<th style="text-align: left;">特点</th>
<th style="text-align: left;">昇腾支持</th>
</tr>
<tr>
<td style="text-align: left;"><strong>PyTorch</strong></td>
<td style="text-align: left;">Meta</td>
<td style="text-align: left;">动态图、易调试、学术界主流</td>
<td style="text-align: left;">torch_npu 适配</td>
</tr>
<tr>
<td style="text-align: left;"><strong>MindSpore</strong></td>
<td style="text-align: left;">华为</td>
<td style="text-align: left;">端边云全场景、自动并行</td>
<td style="text-align: left;">原生支持昇腾</td>
</tr>
<tr>
<td style="text-align: left;"><strong>TensorFlow</strong></td>
<td style="text-align: left;">Google</td>
<td style="text-align: left;">静态图、部署生态完善</td>
<td style="text-align: left;">通过适配层支持</td>
</tr>
</table>

**框架对比详解**：
- **PyTorch**：采用动态计算图（Eager Execution），代码即图，调试方便，可以随时打印中间变量。学术界主流选择，论文中的新模型几乎都用 PyTorch 实现。在昇腾平台上通过 `torch_npu` 插件适配，只需将 `.to('cuda')` 改为 `.to('npu')` 即可运行。
- **MindSpore**：华为自研的深度学习框架，支持端、边、云全场景部署。原生支持昇腾 NPU，提供自动并行、自动微分等高级特性。其核心优势在于软硬一体设计——从芯片指令集到计算图编译进行全方位适配，最大限度释放专用 AI 芯片的算力潜力。
- **TensorFlow**：采用静态计算图（2.x 后默认 Eager），部署生态完善（TF Serving、TF Lite、TF.js）。在昇腾上通过适配层支持，但不如 PyTorch/MindSpore 原生。

> **选择建议**：学术研究和快速原型验证用 PyTorch；昇腾原生开发和极致性能用 MindSpore；工业部署生态用 TensorFlow。通过 ONNX 等开放标准，模型在不同框架间的迁移和部署变得更加便捷。

### 4.2.2 昇腾平台开发流程

```
定义模型 -> .to('npu') -> 训练/推理 -> 部署
```

> 在昇腾平台上，只需将设备从 `cpu`/`cuda` 改为 `npu`，即可享受 NPU 加速！



---

## 4.3 深度学习模型训练开发

深度学习网络模型需要通过大量的数据和计算进行训练，才能获得最终的可用模型。一个训练完成的模型通常包含两部分：**网络结构**（定义了层的类型、连接方式等）和**网络参数**（即训练得到的权重与偏置）。

### 4.3.1 完整案例：手写数字识别

下面在昇腾 NPU 上完成一个完整的训练流程：构建模型 -> 训练 -> 评估。

> 使用随机数据模拟 MNIST，重点演示流程。

**代码说明**：
1. **准备数据**：用 `torch.randn` 生成 800 个训练样本和 200 个测试样本（1×28×28 灰度图），标签为 0~9 的随机整数。
2. **创建模型**：实例化 `SimpleCNN`，使用 SGD 优化器（学习率 0.01，动量 0.9）和交叉熵损失函数。
3. **训练循环**：共 5 个 epoch，batch_size=32。每个 epoch 内随机打乱数据，分批前向→计算损失→反向→更新权重。每个 epoch 结束后在测试集上评估准确率。

**预期结果**：
- `训练集: torch.Size([800, 1, 28, 28]), 测试集: torch.Size([200, 1, 28, 28])`
- 每个 epoch 打印 `loss` 和 `test_acc`，由于使用随机数据（标签与图像无真实关联），测试准确率应在 0.10 左右（10 类随机猜测的期望值）。
- loss 值会逐渐下降（模型在学习拟合随机标签），但 test_acc 不会显著提升（因为测试集标签也是随机的，不存在可学习的规律）。
- `训练完成！`

> **为什么准确率约 0.10？** 因为训练集和测试集的标签都是随机生成的整数（0~9），图像与标签之间不存在真实的映射关系。模型虽然在训练集上能过拟合（loss 下降），但这种"学习"无法泛化到测试集。10 类问题的随机猜测期望准确率为 1/10=0.10。

> **关于全连接层维度**：网络经过 `Conv2d(1→32) → MaxPool(2) → Conv2d(32→64) → MaxPool(2)` 后，空间尺寸从 28→14→7，因此全连接层输入维度为 64×7×7=3136。如果错误地设为 64×14×14=12544，会导致维度不匹配，在 NPU 上会报 `aclnnAddmm` 的 k-axis 错误。



In [ ]:
# ===== 完整训练流程：手写数字识别 =====
import time

# 1. 准备数据（随机模拟 MNIST）
torch.manual_seed(42)
train_x = torch.randn(800, 1, 28, 28).to(device)
train_y = torch.randint(0, 10, (800,)).to(device)
test_x = torch.randn(200, 1, 28, 28).to(device)
test_y = torch.randint(0, 10, (200,)).to(device)
print(f'训练集: {train_x.shape}, 测试集: {test_x.shape}')

# 2. 创建模型、优化器、损失函数
model = SimpleCNN().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

# 3. 训练循环
EPOCHS, BATCH_SIZE = 5, 32
print('\n===== 开始训练 =====')
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(800)
    epoch_loss = 0.0
    for i in range(0, 800, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        out = model(train_x[idx])
        loss = criterion(out, train_y[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    # 评估
    model.eval()
    with torch.no_grad():
        pred = model(test_x).argmax(dim=1)
        acc = (pred == test_y).float().mean().item()
    print(f'Epoch {epoch+1}/{EPOCHS}  loss: {epoch_loss/(800//BATCH_SIZE):.4f}  test_acc: {acc:.4f}')

print('\n训练完成！')


### 4.3.2 NPU vs CPU 性能对比

让我们对比 NPU 和 CPU 在矩阵乘法上的性能差异，直观感受 NPU 加速。

**代码说明**：
- 生成两个 4000×4000 的 FP32 随机矩阵（每个约 61MB），分别在 CPU 和 NPU 上做矩阵乘法。
- CPU 计时直接用 `time.time()`；NPU 计时前后需调用 `torch.npu.synchronize()` 确保异步计算完成后再取时间。
- 最后用 `torch.allclose` 验证 NPU 和 CPU 的计算结果是否一致（容差 rtol=1e-3, atol=1e-3）。

**预期结果**：
- `矩阵大小: 4000x4000，约 61 MB`
- `CPU 矩阵乘法耗时: 约 3000~8000 ms`（取决于 CPU 核数和频率）
- `NPU 矩阵乘法耗时: 约 50~200 ms`（昇腾 910B3 的 AI Core 专门优化矩阵运算）
- `加速比: 约 15~50x`（NPU 针对矩阵乘法有专用算力单元，加速效果显著）
- `结果一致: True`（NPU 和 CPU 的计算结果在浮点误差范围内一致）

> **为什么 NPU 快这么多？** 昇腾 NPU 内置专用的矩阵计算单元（Cube/Matrix Unit），一个时钟周期可以完成大量乘加运算（如 16×16 的矩阵乘法），而 CPU 的通用 ALU 需要逐条指令执行。此外 NPU 有高带宽的片上内存（LM），减少了数据搬运延迟。`synchronize()` 的作用是等待 NPU 异步计算完成——NPU 提交任务后立即返回，不等待计算结束，因此必须显式同步才能准确计时。



In [ ]:
# NPU vs CPU 矩阵乘法性能对比
N = 4000
a = torch.randn(N, N, dtype=torch.float32)
b = torch.randn(N, N, dtype=torch.float32)
print(f'矩阵大小: {N}x{N}，约 {a.numel() * 4 / 1024**2:.0f} MB')

# CPU 计时
start = time.time()
c_cpu = a @ b
cpu_time = time.time() - start
print(f'CPU 矩阵乘法耗时: {cpu_time*1000:.1f} ms')

# NPU 计时
if torch.npu.is_available():
    a_npu = a.npu()
    b_npu = b.npu()
    torch.npu.synchronize()
    start = time.time()
    c_npu = a_npu @ b_npu
    torch.npu.synchronize()
    npu_time = time.time() - start
    print(f'NPU 矩阵乘法耗时: {npu_time*1000:.1f} ms')
    print(f'加速比: {cpu_time/npu_time:.1f}x')
    # 验证结果一致
    match = torch.allclose(c_npu.cpu(), c_cpu, rtol=1e-3, atol=1e-3)
    print(f'结果一致: {match}')
else:
    print('NPU 不可用，跳过 NPU 测试')


---

## 小结

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">知识点</th><th style="text-align: left;">一句话理解</th></tr>
<tr><td style="text-align: left;">CNN</td><td style="text-align: left;">局部连接+权值共享，空间特征提取专家</td></tr>
<tr><td style="text-align: left;">RNN/LSTM</td><td style="text-align: left;">循环连接维护时序记忆，序列建模经典方案</td></tr>
<tr><td style="text-align: left;">Transformer</td><td style="text-align: left;">自注意力实现全局建模+高度并行，大模型基础架构</td></tr>
<tr><td style="text-align: left;">深度可分离卷积</td><td style="text-align: left;">将标准卷积拆为逐通道+逐点，MobileNet 的核心</td></tr>
<tr><td style="text-align: left;">分组卷积+通道混洗</td><td style="text-align: left;">分组减计算，混洗促交流，ShuffleNet 的核心</td></tr>
<tr><td style="text-align: left;">复合缩放</td><td style="text-align: left;">统一缩放深度/宽度/分辨率，EfficientNet 的核心</td></tr>
<tr><td style="text-align: left;">MobileViT</td><td style="text-align: left;">CNN 局部建模 + Transformer 全局建模的混合架构</td></tr>
<tr><td style="text-align: left;">量化</td><td style="text-align: left;">FP32->INT8，体积缩小~4倍</td></tr>
<tr><td style="text-align: left;">剪枝</td><td style="text-align: left;">去除不重要权重/通道，减少冗余</td></tr>
<tr><td style="text-align: left;">知识蒸馏</td><td style="text-align: left;">大模型教小模型，小模型逼近大模型精度</td></tr>
<tr><td style="text-align: left;">图优化</td><td style="text-align: left;">算子融合+常量折叠+内存规划，提升执行效率</td></tr>
<tr><td style="text-align: left;">ONNX</td><td style="text-align: left;">开放标准中间表示，一次训练多处部署</td></tr>
<tr><td style="text-align: left;">昇腾 NPU</td><td style="text-align: left;">一行代码 .to('npu') 即享 AI 计算加速</td></tr>
</table>

---



## 课后练习

请根据本节课程学习内容完成以下题目进行自测。



**第1题**（单选题）CNN 的三大核心设计原则不包括以下哪一项？

- A. 局部连接
- B. 权值共享
- C. 层次化特征提取
- D. 全局注意力



In [ ]:
q1 = ''  # 填入你的选项，如 'D'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')


**第2题**（单选题）Transformer 的核心创新是什么？

- A. 卷积操作
- B. 自注意力机制
- C. 循环连接
- D. 池化操作



In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')


**第3题**（单选题）将模型从 FP32 量化到 INT8，体积大约缩小多少倍？

- A. 2 倍
- B. 4 倍
- C. 8 倍
- D. 16 倍



In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')


**第4题**（单选题）在昇腾平台上，如何将 PyTorch 模型迁移到 NPU 运行？

- A. 重新编写所有代码
- B. 导入 torch_npu 并调用 .to('npu') 或 .npu()
- C. 安装 CUDA 驱动
- D. 无法迁移



In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')


**第5题**（单选题）知识蒸馏的核心思想是什么？

- A. 用大模型教小模型，让小模型逼近大模型精度
- B. 把模型权重置零
- C. 降低数据精度
- D. 增加模型层数



In [ ]:
q5 = ''  # 填入你的选项，如 'A'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')


**第6题**（单选题）LSTM 解决 RNN 什么问题？

- A. 计算速度慢
- B. 梯度消失/爆炸，无法学习长期依赖
- C. 参数太多
- D. 无法并行



In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')


**第7题**（单选题）以下哪个不是模型压缩方法？

- A. 量化
- B. 剪枝
- C. 知识蒸馏
- D. 数据增强



In [ ]:
q7 = ''  # 填入你的选项，如 'D'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')


**第8题**（单选题）ResNet 的核心创新是什么？

- A. 深度可分离卷积
- B. 残差学习框架，解决梯度消失
- C. 自注意力机制
- D. 门控循环



In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')


**第9题**（单选题）MobileNet 使用什么技术减少计算量？

- A. 残差连接
- B. 深度可分离卷积
- C. 自注意力
- D. 双向循环



In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')


**第10题**（单选题）在昇腾 NPU 上计时需要调用什么函数同步？

- A. torch.cuda.synchronize()
- B. torch.npu.synchronize()
- C. time.sleep()
- D. 不需要同步



In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')


**全部作答完成后，运行下方代码查看批改结果：**



In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_01 import grade
grade(globals())


## 参考资料

- [昇腾社区 - CANN 文档](https://hiascend.com/document)
- [PyTorch 官方教程](https://pytorch.org/tutorials/)
- [MindSpore 官方文档](https://www.mindspore.cn/)
- [ONNX 官方文档](https://onnx.ai/)
- [ONNX Runtime 文档](https://onnxruntime.ai/)
